## **Setup**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import os as os
import json as json
import time
import math
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt

# Local
from chatGnT.config import CFG, ensure_dirs
from chatGnT.data import load, preprocess, tokenize, dataloaders
from chatGnT.models import train, evaluate, predict
import chatGnT.utils as utils
from chatGnT.models.transformer import TransformerModel_SingleTask, TransformerModel_MultiTask

ensure_dirs(CFG)

In [ ]:
def load_model_mt(path, model_class):
    checkpoint = torch.load(path, map_location="cpu")

    model = model_class(**checkpoint["config"])
    model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()

    return (
        model,
        checkpoint["vocab_amt"],
        checkpoint["vocab_ingred"],
        checkpoint["config"],
    )

def load_model_st(path, model_class):
    checkpoint = torch.load(path, map_location="cpu")

    model = model_class(**checkpoint["config"])
    model.load_state_dict(checkpoint["model_state_dict"])

    model.eval()

    return (
        model,
        checkpoint["vocab"],
        checkpoint["config"],
    )

In [ ]:
# Load models
model_mt, vocab_mt_amt, vocab_mt_ingred, config_mt = load_model_mt(
    "../outputs/models/multi_task/model.pt", TransformerModel_MultiTask)

model_st, vocab_st, config_st = load_model_st(
    "../outputs/models/single_task/model.pt", TransformerModel_SingleTask)


## **Run with Test User Input**

#### **Single Task Model**

In [ ]:
input_mod = ["<amt>1 oz</amt>", "<ingred>gin</ingred>"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_mt = model_mt.to(device)

inv_vocab_st = {v: k for k, v in vocab_st.items()}

pad_id = vocab_st["<pad>"]

predict.predict_st(model_st, device, pad_id, vocab_st, inv_vocab_st, input_mod)

#### **Multi Task Model**

In [ ]:
input_mod = [("<amt>1 oz</amt>", "<ingred>gin</ingred>")]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_mt = model_mt.to(device)

inv_vocab_mt_amt = {v: k for k, v in vocab_mt_amt.items()}
inv_vocab_mt_ingred = {v: k for k, v in vocab_mt_ingred.items()}

pad_id_amt = vocab_mt_amt["<pad>"]
pad_id_ingred = vocab_mt_ingred["<pad>"]

predict.predict_mt(
    model_mt, device, pad_id_amt, pad_id_ingred, vocab_mt_amt, vocab_mt_ingred,
    inv_vocab_mt_amt, inv_vocab_mt_ingred, input_mod)


## **Visualize Embeddings**